# Imports:

In [6]:
import tensorflow as tf
from tensorflow import keras

physical_devices = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(physical_devices))

if physical_devices:
    for gpu in physical_devices:
        print("Device Details:", gpu)
        
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("Warning: TensorFlow is using the CPU.")

Num GPUs Available:  0


In [9]:
import pandas as pd
import numpy as np
import json
from PIL import Image
import matplotlib.pyplot as plt
# from utils.utils_augmentation import *

In [12]:
# 1. Load the CSVs without 'dx_encoded' (since it's not in the file)
train_df = pd.read_csv('data/augmented_metadata.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path', 'dx']]
val_df = pd.read_csv('data/val_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path', 'dx']]
test_df = pd.read_csv('data/test_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path','dx']]

# 2. Load your mapping dictionary
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

# 3. Re-create the encoded column using the loaded dictionary
val_df["dx_encoded"] = val_df["dx"].map(label2idx)
test_df["dx_encoded"] = test_df["dx"].map(label2idx)

FileNotFoundError: [Errno 2] No such file or directory: 'data/augmented_metadata.csv'

In [5]:
train_df

,image_id,dataset,lesion_id,cleaned_path,dx,dx_encoded
0,ISIC_0028291,vidir_modern,HAM_0006253,./data\HAM10000_cleaned\ISIC_0028291_cleaned.jpg,mel,4
1,ISIC_0028308,rosendahl,HAM_0003752,./data\HAM10000_cleaned\ISIC_0028308_cleaned.jpg,bkl,2
2,ISIC_0028286,vidir_molemax,HAM_0005470,./data\HAM10000_cleaned\ISIC_0028286_cleaned.jpg,nv,5
3,ISIC_0028300,vidir_molemax,HAM_0004092,./data\HAM10000_cleaned\ISIC_0028300_cleaned.jpg,nv,5
4,ISIC_0028292,vidir_molemax,HAM_0006686,./data\HAM10000_cleaned\ISIC_0028292_cleaned.jpg,nv,5
...,...,...,...,...,...,...
7986,ISIC_0029315,rosendahl,HAM_0004780,./data\HAM10000_cleaned\ISIC_0029315_cleaned.jpg,akiec,0
7987,ISIC_0029334,vienna_dias,HAM_0002881,./data\HAM10000_cleaned\ISIC_0029334_cleaned.jpg,nv,5
7988,ISIC_0029335,vidir_molemax,HAM_0005119,./data\HAM10000_cleaned\ISIC_0029335_cleaned.jpg,nv,5
7989,ISIC_0029324,vidir_molemax,HAM_0007380,./data\HAM10000_cleaned\ISIC_0029324_cleaned.jpg,nv,5


In [6]:
train_df = train_df.rename(columns={'cleaned_path': 'image_path'})
val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dataset'] = 'original'
val_df['dataset'] = 'original'
test_df['dataset'] = 'original'

with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

In [7]:
train_df.head()

,image_id,dataset,lesion_id,image_path,dx,dx_encoded
0,ISIC_0028291,original,HAM_0006253,./data\HAM10000_cleaned\ISIC_0028291_cleaned.jpg,mel,4
1,ISIC_0028308,original,HAM_0003752,./data\HAM10000_cleaned\ISIC_0028308_cleaned.jpg,bkl,2
2,ISIC_0028286,original,HAM_0005470,./data\HAM10000_cleaned\ISIC_0028286_cleaned.jpg,nv,5
3,ISIC_0028300,original,HAM_0004092,./data\HAM10000_cleaned\ISIC_0028300_cleaned.jpg,nv,5
4,ISIC_0028292,original,HAM_0006686,./data\HAM10000_cleaned\ISIC_0028292_cleaned.jpg,nv,5


# Modelling - Custom CNN

In [ ]:
N_CLASSES = 7
BATCH_SIZE = 32
class_weights_dict = make_class_weights(aug_train_df)